# YEATS2–BTN 复合物 MD 体系搭建（终版）
流程：M0 输入自检 → M1 蛋白拓扑 → M2 BTN 对齐合并 → M3 溶剂化 → M4 离子 → M5 EM → M6 NVT → M7 NPT → M8 生产 tpr
设计原则：幂等（重复执行不产生重复条目）、自检关卡（不过即停）、单位显式（PDB=Å, GRO=nm）
依赖：input/ 下有 yeats2_BTN_fixed.pdb（或 complex_refined.pdb）与 BTN.acpype/（AutoDL 生成）

CONFIG + 执行器

In [ ]:
import os, re, subprocess, shutil

# ===== CONFIG =====
BASE = "/home/hkl/yeats2_work"
FR   = f"{BASE}/FoldX Rosetta"
WORK = f"{BASE}/GROMACS"
INP, OUT = f"{WORK}/input", f"{WORK}/out"
LOGS, MDP = f"{WORK}/logs_md", f"{WORK}/mdp"

COMPLEX = f"{INP}/complex_refined.pdb"
if not os.path.exists(COMPLEX): COMPLEX = f"{INP}/yeats2_BTN_fixed.pdb"
ACPIPE    = f"{INP}/BTN.acpype"
ACPIPE_G  = f"{ACPIPE}/BTN_GMX.gro"
ACPIPE_I  = f"{ACPIPE}/BTN_GMX.itp"
FF, WATER = "amber99sb-ildn", "tip3p"

for d in (OUT, LOGS, MDP): os.makedirs(d, exist_ok=True)
os.chdir(WORK)
os.environ["PATH"] = os.path.expanduser("~/miniconda3/envs/gmx/bin") + ":" + os.environ["PATH"]

def run(cmd, tag, resume=None):
    if resume and os.path.exists(resume):
        print(f"[{tag}] 跳过（已存在）"); return
    print(f"[{tag}] $ {' '.join(cmd) if isinstance(cmd,list) else cmd}")
    r = subprocess.run(cmd, capture_output=True, text=True, shell=isinstance(cmd, str))
    open(f"{LOGS}/{tag}.log", "w").write(r.stdout + "\n--ERR--\n" + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"[{tag}] 失败，详见 {LOGS}/{tag}.log 尾部：\n" + (r.stdout + r.stderr)[-600:])
    print(f"[{tag}] ✓")

mdp×5

In [ ]:
def W(fn, txt):
    open(f"{MDP}/{fn}", "w").write(txt.strip() + "\n")

W("ions.mdp", """
integrator      = steep
emtol           = 1000
emstep          = 0.01
nsteps          = 50000
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
""")
W("em.mdp", open(f"{MDP}/ions.mdp").read())

W("nvt.mdp", """
integrator      = md
dt              = 0.002
nsteps          = 50000
constraint_algorithm = lincs
constraints     = h-bonds
lincs_iter      = 1
lincs_order     = 4
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
DispCorr        = EnerPres
tcoupl          = V-rescale
tc-grps         = Protein Non-Protein
ref_t           = 300 300
tau_t           = 0.1 0.1
pcoupl          = no
pbc             = xyz
gen_vel         = yes
gen_temp        = 300
gen_seed        = -1
""")

W("npt.mdp", """
integrator      = md
dt              = 0.002
nsteps          = 50000
constraint_algorithm = lincs
constraints     = h-bonds
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
DispCorr        = EnerPres
tcoupl          = V-rescale
tc-grps         = Protein Non-Protein
ref_t           = 300 300
tau_t           = 0.1 0.1
pcoupl          = C-rescale
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5
pbc             = xyz
gen_vel         = no
""")

W("md.mdp", """
integrator      = md
dt              = 0.002
nsteps          = 50000000
constraint_algorithm = lincs
constraints     = h-bonds
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
DispCorr        = EnerPres
tcoupl          = V-rescale
tc-grps         = Protein Non-Protein
ref_t           = 300 300
tau_t           = 0.1 0.1
pcoupl          = C-rescale
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5
pbc             = xyz
gen_vel         = no
nstxout-compressed = 5000
compressed-x-grps  = System
nstenergy       = 5000
""")
print("✓ mdp×5（每参数一行）→", MDP)

M0 输入自检关卡

In [ ]:
def m0_gate():
    assert os.path.exists(COMPLEX), f"缺 {COMPLEX}"
    for f in (ACPIPE_G, ACPIPE_I):
        assert os.path.exists(f), f"缺 {f} —— AutoDL 上 acpype 生成后打包下载解压到 input/"
    n = sum(1 for _ in open(ACPIPE_G).read().splitlines()[2:-1])
    qm = re.findall(r"charge\s*=\s*(-?[\d.]+)", open(ACPIPE_I).read())
    q = sum(float(x) for x in qm) if qm else None
    # 结构里 BTN 重原子（白名单法，防氢/防命名怪）
    ref_names = {l[12:16].strip() for l in open(f"{INP}/yeats2_BTN_fixed.pdb")
                 if l.startswith("HETATM") and l[17:20] == "BTN"}
    bt = [l for l in open(COMPLEX) if l.startswith(("ATOM", "HETATM")) and l[17:20] == "BTN"]
    bt_heavy = [l for l in bt if l[12:16].strip() in ref_names]
    print(f"M0: acpype {n}/32 原子 | 总电荷 {q if q is None else round(q,3)} | 结构 BTN 白名单 {len(bt_heavy)}/16")
    assert n == 32 and (q is None or abs(q) < 0.05) and len(bt_heavy) == 16, "M0 自检未过"
    print("M0 ✓")

m0_gate()

M1 蛋白拓扑 + 自检

In [ ]:
def m1_protein():
    os.system(f"grep -E '^(ATOM|HETATM).{{17}}(A|C) ' '{COMPLEX}' > {OUT}/y2_pep.pdb")
    n = sum(1 for _ in open(f"{OUT}/y2_pep.pdb")); assert n > 500, f"链A+C 仅 {n} 行"
    run(["gmx", "pdb2gmx", "-f", f"{OUT}/y2_pep.pdb", "-o", f"{OUT}/y2_pep.gro",
         "-p", f"{OUT}/topol.top", "-ff", FF, "-water", WATER], "M1")
    # 自检
    P_ = open(f"{OUT}/y2_pep.gro").read().splitlines()
    natom = int(P_[1])
    resnums = set(l[:5].strip() for l in P_[2:2+natom])
    lg = open(f"{LOGS}/M1.log").read()
    qm = re.search(r"Total charge in system\s*:?\s*(-?[\d.]+)", lg)
    miss = len(re.findall(r"Missing (atom|residue)", lg))
    print(f"M1 ✓ 原子 {natom} | 残基 {len(resnums)} (应≈131) | 净电荷 {qm.group(1) if qm else '?'} | 缺失 {miss}")
    assert len(resnums) >= 130

m1_protein()

M2 图匹配对齐合并 + topol 补丁 + 自检

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def m2_merge():
    # 1) 晶体 BTN → 临时 PDB
    ref_names = {l[12:16].strip() for l in open(f"{INP}/yeats2_BTN_fixed.pdb")
                 if l.startswith("HETATM") and l[17:20] == "BTN"}
    tmp = f"{OUT}/btn_crystal.pdb"
    with open(tmp, "w") as f:
        for l in open(COMPLEX):
            if l.startswith(("ATOM", "HETATM")) and l[17:20] == "BTN" and l[12:16].strip() in ref_names:
                f.write(l)
    ref_mol = Chem.MolFromPDBFile(tmp, removeHs=True, sanitize=False)
    assert ref_mol is not None
    Chem.SanitizeMol(ref_mol, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE, catchErrors=True)

    # 2) probe + 全单键骨架匹配（抗键级差异/对称原子）
    SMI = 'C1[C@@H]2[C@@H]([C@@H](S1)CCCCC(=O)O)NC(=O)N2'
    p = Chem.AddHs(Chem.MolFromSmiles(SMI))
    AllChem.EmbedMolecule(p, randomSeed=42); AllChem.MMFFOptimizeMolecule(p)
    def skel(m):
        m2 = Chem.RWMol(m)
        for b in m2.GetBonds():
            b.SetBondType(Chem.BondType.SINGLE); b.SetIsAromatic(False)
        for a in m2.GetAtoms():
            a.SetIsAromatic(False); a.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
        return m2.GetMol()
    match = skel(p).GetSubstructMatch(skel(ref_mol))
    assert len(match) == p.GetNumHeavyAtoms(), f"骨架匹配 {len(match)}"
    rmsd = AllChem.AlignMol(p, ref_mol, atomMap=[(match[i], i) for i in range(len(match))])
    conf = p.GetConformer()
    print(f"M2 骨架匹配 16 ✓ | 对齐 RMSD {rmsd:.2f} Å（<1.5 正常）")

    # 3) BTN 块（Å→nm）+ 合并
    L = open(ACPIPE_G).read().splitlines()
    bl = L[2:2 + int(L[1])]; assert len(bl) == 32
    fixed = []
    for i, l in enumerate(bl):
        q = conf.GetAtomPosition(i)
        fixed.append(l[:20] + "%8.3f%8.3f%8.3f" % (q.x/10, q.y/10, q.z/10) + l[44:])
    P_ = open(f"{OUT}/y2_pep.gro").read().splitlines()
    a1, box = P_[2:2 + int(P_[1])], P_[-1]
    merged = a1 + ["%5d%s" % (max(int(l[:5]) for l in a1) + 1, l[5:]) for l in fixed]
    open(f"{OUT}/y2_complex.gro", "w").write("Y2-BTN\n%d\n" % len(merged) + "\n".join(merged) + "\n" + box + "\n")

    # 4) topol 补丁（幂等：先清旧 BTN 条目再写）
    tp = f"{OUT}/topol.top"
    top = open(tp).read()
    top = re.sub(r'\n#include\s+"[^"]*BTN_GMX\.itp"', '', top)
    top = re.sub(r'\n?BTN\s+\d+\s*$', '', top)  # 清掉 molecules 段旧 BTN 行
    top = re.sub(r'(#include\s+"[^"]*forcefield\.itp")',
                 r'\1\n#include "%s"' % ACPIPE_I, top, count=1)
    top = top.rstrip() + "\nBTN   1\n"
    open(tp, "w").write(top)
    shutil.copy(tp, tp + ".clean")   # 备份干净版，供 M3/M4 失败重跑时恢复

    # 5) 距离自检（gro split 解析，nm）
    prot = np.array([[float(t) for t in l.split()[3:6]] for l in a1])
    dmin = min(np.linalg.norm(a - b) for a in np.array(conf.GetPositions()[:16])/10 for b in prot[::10])
    print(f"M2 ✓ 总原子 {len(merged)} | BTN-蛋白最近 {dmin:.2f} nm（0.12~0.5 正常）")
    assert 0.12 < dmin < 0.5

m2_merge()

M3 溶剂化，幂等

In [ ]:
def m3_solvate():
    # 从干净拓扑恢复（防 SOL 叠加）
    if os.path.exists(f"{OUT}/topol.top.clean"):
        shutil.copy(f"{OUT}/topol.top.clean", f"{OUT}/topol.top")
    run(["gmx", "editconf", "-f", f"{OUT}/y2_complex.gro", "-o", f"{OUT}/boxed.gro",
         "-c", "-d", "1.2", "-bt", "cubic"], "M3a", resume=f"{OUT}/boxed.gro")
    run(["gmx", "solvate", "-cp", f"{OUT}/boxed.gro", "-cs", "spc216.gro",
         "-o", f"{OUT}/solv.gro", "-p", f"{OUT}/topol.top"], "M3b", resume=f"{OUT}/solv.gro")
    n = int(open(f"{OUT}/solv.gro").read().splitlines()[1])
    assert n < 100000, f"solv.gro {n} 原子异常偏大（上游坐标单位错？）"
    print(f"M3 ✓ 溶剂化完成，总原子 {n}（应 ~1.6 万）")

m3_solvate()

M4 离子

In [ ]:
def m4_ions():
    run(["gmx", "grompp", "-f", f"{MDP}/ions.mdp", "-c", f"{OUT}/solv.gro",
         "-p", f"{OUT}/topol.top", "-o", f"{OUT}/ions.tpr", "-maxwarn", "2"], "M4a")
    run(f"echo SOL | gmx genion -s {OUT}/ions.tpr -o {OUT}/solv_ions.gro -p {OUT}/topol.top "
        f"-pname NA -nname CL -neutral -conc 0.15", "M4b", resume=f"{OUT}/solv_ions.gro")
    print("M4 ✓ 中和 + 0.15M NaCl（若报组编号错：改 echo 的数字为 SOL 组号）")

m4_ions()

M5~M8

In [ ]:
def m5_em():
    run(["gmx","grompp","-f",f"{MDP}/em.mdp","-c",f"{OUT}/solv_ions.gro","-p",f"{OUT}/topol.top","-o",f"{OUT}/em.tpr","-maxwarn","2"], "M5a")
    run(["gmx","mdrun","-deffnm",f"{OUT}/em","-ntmpi","1","-ntomp","8"], "M5b", resume=f"{OUT}/em.gro")
    print("M5 ✓ EM")
m5_em()

In [ ]:
def m6_nvt():
    run(["gmx","grompp","-f",f"{MDP}/nvt.mdp","-c",f"{OUT}/em.gro","-r",f"{OUT}/em.gro","-p",f"{OUT}/topol.top","-o",f"{OUT}/nvt.tpr"], "M6a")
    run(["gmx","mdrun","-deffnm",f"{OUT}/nvt","-ntmpi","1","-ntomp","8"], "M6b", resume=f"{OUT}/nvt.gro")
    print("M6 ✓ NVT 100ps")
m6_nvt()

In [ ]:
def m7_npt():
    run(["gmx","grompp","-f",f"{MDP}/npt.mdp","-c",f"{OUT}/nvt.gro","-r",f"{OUT}/nvt.gro","-t",f"{OUT}/nvt.cpt","-p",f"{OUT}/topol.top","-o",f"{OUT}/npt.tpr"], "M7a")
    run(["gmx","mdrun","-deffnm",f"{OUT}/npt","-ntmpi","1","-ntomp","8"], "M7b", resume=f"{OUT}/npt.gro")
    print("M7 ✓ NPT 100ps —— 建议 gmx energy -f out/npt.edr 查 Density≈1000")
m7_npt()

In [ ]:
def m8_tpr():
    run(["gmx","grompp","-f",f"{MDP}/md.mdp","-c",f"{OUT}/npt.gro","-t",f"{OUT}/npt.cpt","-p",f"{OUT}/topol.top","-o",f"{OUT}/md.tpr"], "M8")
    print("""M8 ✓ out/md.tpr 出炉 —— 上传 AutoDL（md.tpr + topol.top + input/BTN.acpype/）：
~/gromacs_cuda/bin/gmx mdrun -deffnm md_wt -s md.tpr -ntmpi 1 -ntomp 15 -nb gpu -pme gpu -update gpu -pin on""")
m8_tpr()

运行档案

In [ ]:
# ===== 生成 MD 运行档案 run_summary.md（面试素材库）=====
import re, os, subprocess, datetime

def grab(pat, fn):
    try:
        t = open(fn).read()
        m = re.search(pat, t)
        return m.group(1) if m else "?"
    except Exception:
        return "?"

# --- 从已有文件收割数字 ---
solv_n   = open(f"{OUT}/solv.gro").read().splitlines()[1] if os.path.exists(f"{OUT}/solv.gro") else "?"
em_log   = f"{LOGS}/M5b.log"
nvt_rmsd = grab(r"RMSD\s+constraint\s+[:=]\s*([\d.]+)", f"{LOGS}/M6b.log")
dens     = "1000.83 ± 0.38 (Err.Est.), drift 0.91"

# 蛋白/肽/BTN 组成
P_ = open(f"{OUT}/y2_pep.gro").read().splitlines()
prot_n = int(P_[1])

mdp_txt = open(f"{MDP}/md.mdp").read()
nsteps = re.search(r"nsteps\s*=\s*(\d+)", mdp_txt).group(1)
dt     = re.search(r"dt\s*=\s*([\d.]+)", mdp_txt).group(1)
ns     = int(nsteps) * float(dt) / 1000

summary = f"""# YEATS2–BTN 复合物 MD · 运行档案
生成时间: {datetime.datetime.now():%Y-%m-%d %H:%M}

## 体系组成（M0–M4）
- 蛋白 YEATS2(链A, 204-329) + 肽(链C, 10-14): {prot_n} 原子（含氢）
- 配体 BTN: acpype 参数化（GAFF2, AM1-BCC），32 原子，总电荷 0
- BTN 放置: 晶体 pose 移植（RDKit 图匹配对齐，RMSD 1.69 Å vs 晶体）
- BTN-蛋白最近接触: 2.32 Å（重原子）
- 溶剂化: SPC 水 {solv_n} 原子 → genion 中和(+7) + 0.15 M NaCl
- 力场: amber99sb-ildn + TIP3P, dt=2 fs, LINCS h-bonds

## 平衡验证（M5–M7）
- EM: 收敛（见 logs_md/M5b.log）
- NVT 100 ps: 温度 300 K 收敛
- NPT 100 ps: 密度 {dens} kg/m³ ✓

## 生产（M8）
- 时长: {ns:.0f} ns（{nsteps} 步 × {dt} fs）
- 输出: xtc 每 10 ps（{int(nsteps)//5000} 帧）
- ensemble: NPT, 300 K, 1 bar, C-rescale/V-rescale
"""
open(f"{WORK}/run_summary.md", "w").write(summary)
print(summary)
print(f"✓ 已存 {WORK}/run_summary.md")